# 🧩 01 · 图像编码与 ViT

> 多模态的"眼睛"是**视觉编码器**。本节手写 **ViT（Vision Transformer）** 的核心步骤：
> patch 切分 → 展平嵌入 → 位置编码 → 自注意力 → 分类头，并可视化注意力热力图思想。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 图像怎么变成序列？ | Patch 切分 + 线性投影 |
| 2 | 位置信息怎么给？ | 位置编码（可学习/正弦） |
| 3 | Transformer 怎么处理图像？ | 自注意力 |
| 4 | 分类怎么做？ | CLS token + 分类头 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 构造一张"图像"并切 Patch

In [2]:
H, W, C = 32, 32, 3
img = rng.uniform(0, 1, (H, W, C))
patch_size = 8
n_patches_h, n_patches_w = H // patch_size, W // patch_size
n_patches = n_patches_h * n_patches_w
patch_dim = patch_size * patch_size * C

# 切 patch：reshape 技巧
patches = (img.reshape(n_patches_h, patch_size, n_patches_w, patch_size, C)
              .transpose(0, 2, 1, 3, 4)
              .reshape(n_patches, patch_dim))
print(f'图像 {H}x{W}x{C} -> {n_patches_h}x{n_patches_w} 网格 -> {n_patches} 个 patch，每 patch {patch_dim} 维')
assert patches.shape == (n_patches, patch_dim)
print('✅ patch 切分完成')

图像 32x32x3 -> 4x4 网格 -> 16 个 patch，每 patch 192 维
✅ patch 切分完成


## 2️⃣ Patch Embedding：线性投影 + 位置编码

In [3]:
emb_dim = 16
W_proj = rng.normal(0, 0.1, (patch_dim, emb_dim))
patch_emb = patches @ W_proj                      # (N_patches, emb_dim)
pos_emb = rng.normal(0, 0.1, (n_patches, emb_dim))  # 可学习位置编码（这里是随机初始化）
x = patch_emb + pos_emb
print('patch embedding:', patch_emb.shape, '+ 位置编码 ->', x.shape)
print('➡️ 真实 ViT：投影矩阵与位置编码都是可学习参数，与 Transformer 一起训练')

patch embedding: (16, 16) + 位置编码 -> (16, 16)
➡️ 真实 ViT：投影矩阵与位置编码都是可学习参数，与 Transformer 一起训练


## 3️⃣ 自注意力（简化手写）

In [4]:
def self_attention(x, Wq, Wk, Wv, scale):
    # x: (N, d) -> 每个 token 的 Q/K/V
    Q = x @ Wq; K = x @ Wk; V = x @ Wv
    scores = Q @ K.T / scale                # (N, N) 注意力分数
    attn = np.exp(scores - scores.max(axis=1, keepdims=True))
    attn = attn / attn.sum(axis=1, keepdims=True)
    out = attn @ V
    return out, attn

d = emb_dim
Wq = rng.normal(0, 0.1, (d, d)); Wk = rng.normal(0, 0.1, (d, d)); Wv = rng.normal(0, 0.1, (d, d))
out, attn = self_attention(x, Wq, Wk, Wv, scale=np.sqrt(d))
print('注意力矩阵:', attn.shape, '行和=1:', np.allclose(attn.sum(axis=1), 1))
print('输出:', out.shape)
assert np.allclose(attn.sum(axis=1), 1)
print('✅ 自注意力前向正确（行归一化）')

注意力矩阵: (16, 16) 行和=1: True
输出: (16, 16)
✅ 自注意力前向正确（行归一化）


## 4️⃣ 注意力热力图可视化（patch 间关系）

In [5]:
plt.figure(figsize=(6, 5))
plt.imshow(attn, cmap='viridis')
plt.colorbar(label='注意力权重')
plt.xlabel('被关注的 patch')
plt.ylabel('查询 patch')
plt.title('Patch 间注意力权重（示意）')
plt.show()
print('➡️ 真实 ViT 的注意力图能揭示模型在看图像的哪个区域')

➡️ 真实 ViT 的注意力图能揭示模型在看图像的哪个区域


C:\Users\24260\AppData\Local\Temp\ipykernel_43904\3997120690.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5️⃣ CLS token 与分类头

In [6]:
# 在序列最前面加一个可学习的 CLS token，用它聚合整张图信息做分类
cls_token = rng.normal(0, 0.05, (1, d))
x_cls = np.vstack([cls_token, x])
print('加 CLS 后序列长度:', x_cls.shape[0], '（真实 ViT 也是 N_patches+1）')

# 简化：取 CLS 位置的输出经分类头（实际是过个线性层）
n_cls_mlp = 10
W_head = rng.normal(0, 0.1, (d, n_cls_mlp))
logits = x_cls[0] @ W_head
print('分类 logits:', logits)
print('➡️ 面试点：为什么用 CLS 而非平均池化？（可学习、等价于注意力汇聚；ViT 可二者皆试）')

加 CLS 后序列长度: 17 （真实 ViT 也是 N_patches+1）
分类 logits: [8.01231441e-03 3.78853840e-05 1.90915930e-03 2.51256415e-02
 1.69157548e-03 2.70834465e-02 1.85250149e-02 4.61132069e-03
 1.58420505e-02 5.59437678e-03]
➡️ 面试点：为什么用 CLS 而非平均池化？（可学习、等价于注意力汇聚；ViT 可二者皆试）


## 6️⃣ CNN vs ViT（面试对比表）

In [7]:
print('''| 维度 | CNN | ViT |
|---|---|---|
| 局部性 | 天生具备（卷积核） | 靠 attention 学 |
| 归纳偏置 | 强（平移不变/局部） | 弱（数据量需求大） |
| 数据需求 | 少 | 多（大数据+预训练） |
| 长程依赖 | 靠加深感受野 | 天然全局（attention） |
| 扩展性 | 好 | 更好（大模型时代主力） |
| 计算量 | 与分辨率线性相关 | 与 patch 数平方（attention） |
''')

| 维度 | CNN | ViT |
|---|---|---|
| 局部性 | 天生具备（卷积核） | 靠 attention 学 |
| 归纳偏置 | 强（平移不变/局部） | 弱（数据量需求大） |
| 数据需求 | 少 | 多（大数据+预训练） |
| 长程依赖 | 靠加深感受野 | 天然全局（attention） |
| 扩展性 | 好 | 更好（大模型时代主力） |
| 计算量 | 与分辨率线性相关 | 与 patch 数平方（attention） |



## 📝 自测清单

- [ ] 能手写 patch 切分（reshape trick）
- [ ] 能解释位置编码的作用与两种方式
- [ ] 能手写简化自注意力并验证行归一化
- [ ] 能解释 CLS token 的作用
- [ ] 能背出 CNN vs ViT 六维对比